In [1]:
# Install required libraries
!pip install transformers langchain langchain-community gradio sentence-transformers faiss-cpu huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [26]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
import os
path = '/content/drive/MyDrive/AI PROJECTS/SentiBot/training.1600000.processed.noemoticon.csv'
print(os.path.exists(path))

True


In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',
                 encoding='latin-1',
                 header=None)

# Name the columns
df.columns = ['sentiment', 'id', 'date', 'query', 'user', 'text']

# Convert sentiment: 0 = negative, 4 = positive
df['sentiment'] = df['sentiment'].map({0: 'negative', 4: 'positive'})

# Preview
print(df.shape)
print(df.head())

(1600000, 6)
  sentiment          id                          date     query  \
0  negative  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1  negative  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2  negative  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3  negative  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4  negative  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


In [3]:
# Take a smaller sample to work faster
df_sample = df.sample(n=10000, random_state=42)

# Keep only what we need
df_sample = df_sample[['sentiment', 'text']].reset_index(drop=True)

# Check balance
print(df_sample['sentiment'].value_counts())
print("\nSample tweet:")
print(df_sample['text'][0])

sentiment
negative    5004
positive    4996
Name: count, dtype: int64

Sample tweet:
@chrishasboobs AHHH I HOPE YOUR OK!!! 


In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings for all tweets
print("Creating embeddings... this may take a few minutes")
texts = df_sample['text'].tolist()
embeddings = embedder.encode(texts, show_progress_bar=True)

# Build FAISS index (this is the RAG part)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print(f"\nRAG index built with {index.ntotal} tweets!")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings... this may take a few minutes


Batches:   0%|          | 0/313 [00:00<?, ?it/s]


RAG index built with 10000 tweets!


In [5]:
from transformers import pipeline

# Load sentiment analysis model
print("Loading sentiment model...")
sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0  # use GPU
)

print("Sentiment model loaded!")

Loading sentiment model...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Sentiment model loaded!


In [6]:
def analyze_sentiment(text):
    result = sentiment_model(text[:512])[0]
    label = result['label'].lower()
    score = round(result['score'] * 100, 2)

    if 'positive' in label:
        sentiment = 'Positive'
        emoji = '😊'
    elif 'negative' in label:
        sentiment = 'Negative'
        emoji = '😞'
    else:
        sentiment = 'Neutral'
        emoji = '😐'

    return sentiment, score, emoji

def find_similar_tweets(query, top_k=3):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)

    similar = []
    for i, idx in enumerate(indices[0]):
        similar.append({
            'tweet': texts[idx],
            'sentiment': df_sample['sentiment'][idx],
            'distance': round(float(distances[0][i]), 2)
        })
    return similar

# Test it
test_text = "I love this beautiful day!"
sentiment, score, emoji = analyze_sentiment(test_text)
print(f"Text: {test_text}")
print(f"Sentiment: {sentiment} {emoji} (Confidence: {score}%)")
print("\nSimilar tweets from dataset:")
for t in find_similar_tweets(test_text):
    print(f"- [{t['sentiment']}] {t['tweet'][:80]}")

Text: I love this beautiful day!
Sentiment: Positive 😊 (Confidence: 98.22%)

Similar tweets from dataset:
- [positive] most beautiful day ever, loving it 
- [positive] It's a beautiful day 
- [positive] A beautiful day in florida . . . great day to hit the road, heading south for bu


In [11]:
import gradio as gr

def sentibot(user_message, history):
    # Analyze sentiment
    sentiment, score, emoji = analyze_sentiment(user_message)

    # Find similar tweets using RAG
    similar_tweets = find_similar_tweets(user_message, top_k=3)

    # Build response
    response = "**Sentiment Analysis Result:**\n"
    response += f"Sentiment: **{sentiment}** {emoji}\n"
    response += f"Confidence: **{score}%**\n\n"
    response += "**Similar tweets from dataset:**\n"

    for i, t in enumerate(similar_tweets, 1):
        response += f"{i}. [{t['sentiment']}] {t['tweet'][:100]}\n"

    return response

# Launch chatbot
demo = gr.ChatInterface(
    fn=sentibot,
    title="SentiBot - Social Media Sentiment Analyzer",
    description="Type any social media post and I'll analyze its sentiment using AI!",
    examples=[
        "I love this amazing product!",
        "This is the worst day of my life",
                "Just had coffee this morning"
    ],
    type="messages"
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d879712bd4c5ad61de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
!pip install gradio[oauth] huggingface_hub

In [14]:
from huggingface_hub import notebook_login
notebook_login()

In [21]:
app_code = """import gradio as gr
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd

embedder = SentenceTransformer('all-MiniLM-L6-v2')
sentiment_model = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

texts = []
embeddings_list = []
index = None

def analyze_sentiment(text):
    result = sentiment_model(text[:512])[0]
    label = result['label'].lower()
    score = round(result['score'] * 100, 2)
    if 'positive' in label:
        return 'Positive', score, '😊'
    elif 'negative' in label:
        return 'Negative', score, '😞'
    else:
        return 'Neutral', score, '😐'

def sentibot(user_message, history):
    sentiment, score, emoji = analyze_sentiment(user_message)
    response = "Sentiment Analysis Result:\\n"
    response += "Sentiment: " + sentiment + " " + emoji + "\\n"
    response += "Confidence: " + str(score) + "%"
    return response

demo = gr.ChatInterface(
    fn=sentibot,
    title="SentiBot - Social Media Sentiment Analyzer",
    description="Type any social media post to analyze its sentiment!",
    type="messages"
)

demo.launch()
"""

with open("/content/app.py", "w") as f:
    f.write(app_code)

print("app.py created!")

app.py created!


In [22]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj="/content/app.py",
    path_in_repo="app.py",
    repo_id="UmmeRubabb/sentibot",
    repo_type="space"
)

print("Uploaded successfully!")

Uploaded successfully!


In [23]:
requirements = """transformers
sentence-transformers
faiss-cpu
numpy
pandas
torch
"""

with open("/content/requirements.txt", "w") as f:
    f.write(requirements)

api.upload_file(
    path_or_fileobj="/content/requirements.txt",
    path_in_repo="requirements.txt",
    repo_id="UmmeRubabb/sentibot",
    repo_type="space"
)

print("Requirements uploaded!")

Requirements uploaded!


In [24]:
app_code = """import gradio as gr
from transformers import pipeline
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
sentiment_model = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

def analyze_sentiment(text):
    result = sentiment_model(text[:512])[0]
    label = result['label'].lower()
    score = round(result['score'] * 100, 2)
    if 'positive' in label:
        return 'Positive', score, '😊'
    elif 'negative' in label:
        return 'Negative', score, '😞'
    else:
        return 'Neutral', score, '😐'

def sentibot(user_message, history):
    sentiment, score, emoji = analyze_sentiment(user_message)
    response = "Sentiment Analysis Result:\\n"
    response += "Sentiment: " + sentiment + " " + emoji + "\\n"
    response += "Confidence: " + str(score) + "%"
    return response

demo = gr.ChatInterface(
    fn=sentibot,
    title="SentiBot - Social Media Sentiment Analyzer",
    description="Type any social media post to analyze its sentiment!"
)

demo.launch()
"""

with open("/content/app.py", "w") as f:
    f.write(app_code)

from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="/content/app.py",
    path_in_repo="app.py",
    repo_id="UmmeRubabb/sentibot",
    repo_type="space"
)
print("Updated successfully!")

Updated successfully!
